# Day 4 模块 2：多种算法对比

同一套 `X_train / X_test`，换不同模型，看测试集 R² / MAE / 训练耗时。

> 数字以**你电脑跑出来的为准**。下面表格是本课数据上的**大致量级**，用来建立预期，不是标准答案满分线。


In [ ]:
from pathlib import Path
import pandas as pd

candidate_paths = [
    Path('day4_cafe_sales.csv'),
    Path('day4') / 'day4_cafe_sales.csv',
    Path('教学课程') / 'day4' / 'day4_cafe_sales.csv',
]
for path in candidate_paths:
    if path.exists():
        csv_path = path
        break
else:
    raise FileNotFoundError('未找到 day4_cafe_sales.csv')

df = pd.read_csv(csv_path)
print(csv_path.resolve())
print('shape:', df.shape)
df.head()


In [ ]:
# 准备特征 X 和目标 y（与 Day 3 同一套，方便对比）
# 特征列：可能影响营收的输入
feature_cols = [
    'price', 'promotion', 'is_weekend', 'temp',
    'quality', 'competitors', 'holiday', 'location',
]
# 天气文字 → 数字分（晴最好，大雨最差）
weather_score_map = {'晴': 1.0, '多云': 0.8, '阴': 0.6, '小雨': 0.4, '大雨': 0.3}
df = df.copy()
df['weather_score'] = df['weather_label'].map(weather_score_map)

X = df[feature_cols + ['weather_score']].copy()  # 特征表
y = df['sales'].copy()  # 目标：营收

print('特征列:', list(X.columns))
print('样本数:', len(X))
X.head()


In [ ]:
from sklearn.model_selection import train_test_split

# test_size=0.2：约 20% 当测试集；random_state=42：随机种子，结果可复现
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print('训练集行数:', len(X_train))
print('测试集行数:', len(X_test))


## 1. 今天上场的模型（先混个脸熟）

| 模型 | 一句话 | 今天代码 |
| --- | --- | --- |
| 线性回归 | 一条（多维）直线从特征猜营收 | `LinearRegression` |
| 决策树 | 一串 if | `DecisionTreeRegressor` |
| 随机森林 | 很多树取平均 | `RandomForestRegressor` |
| K 近邻 (KNN) | 找最像的几天，看它们营收 | `KNeighborsRegressor` |
| XGBoost 等 | 另一类「加成」树方法，往往更强也更绕 | **知道名字**；环境没有库可不装 |

### 和 `Y = KX + B` 的关系

线性回归就是把「一条直线」推广到很多个特征：每个特征一个系数，再加一个截距。
决策树 / 森林则是 if 规则，不一定是直线关系。


## 2. 统一训练与打分

下面循环里每个模型都：`fit` → 测试集预测 → R² / MAE / 耗时。


In [ ]:
import time
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import r2_score, mean_absolute_error

models = {
    '线性回归': LinearRegression(),
    '决策树(深度3)': DecisionTreeRegressor(max_depth=3, random_state=42),
    '决策树(深度8)': DecisionTreeRegressor(max_depth=8, random_state=42),
    '随机森林': RandomForestRegressor(
        n_estimators=100, max_depth=8, random_state=42, n_jobs=-1
    ),
    'KNN(k=5)': KNeighborsRegressor(n_neighbors=5),
}

rows = []
for name, model in models.items():
    t0 = time.time()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    elapsed = time.time() - t0
    rows.append({
        '模型': name,
        '训练R²': round(model.score(X_train, y_train), 3),
        '测试R²': round(r2_score(y_test, y_pred), 3),
        '测试MAE': round(mean_absolute_error(y_test, y_pred), 1),
        '耗时秒': round(elapsed, 3),
    })

result = pd.DataFrame(rows).sort_values('测试R²', ascending=False)
result


## 3. 怎么读这张表

1. **先看测试 R²**（成绩单），再看训练 R²（会不会背题）
2. 决策树深度 8：训练很高、测试掉一截 → 过拟合苗头
3. KNN：在这份特征上往往较弱（距离尺度、维度等因素）
4. 线性回归：在**这份模拟数据**上可能出奇地好——说明数据里线性成分不弱；**换更弯的业务数据时，树模型常常更稳**
5. 随机森林：测试表现稳、还能吐特征重要性，**本项目系统默认用它**

### 项目为什么仍常选随机森林？

| 维度 | 说明 |
| --- | --- |
| 精度 | 多数表格任务上够用、稳 |
| 可解释 | 有 `feature_importances_`（Day 3） |
| 调参 | 有几个旋钮即可，不必一上来最复杂 |
| 工程 | Day 5 Web 系统里直接调用 sklearn 森林 |

> 「某次线性回归分数更高」≠「永远只用直线」。选模型还要看稳定性、可解释、以后加非线性规律时是否吃得消。


## 4. 可选：画测试 R² 柱状图


In [ ]:
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

plot_df = result.sort_values('测试R²')
plt.figure(figsize=(8, 4))
plt.barh(plot_df['模型'], plot_df['测试R²'], color='#6f4e37')
plt.xlabel('测试集 R²')
plt.title('同一数据、不同模型')
plt.tight_layout()
plt.show()


## 5. 关于 XGBoost / LightGBM（只听不装也行）

- 也是树相关家族，比赛和工业里很常见
- 往往更强一点，但**概念和调参更重**
- 200 天小表：随机森林通常已经够用 → 「杀鸡不必先上牛刀」

若机器已安装 `xgboost`，可自行加一行对比；**没装不要为了今天强行 conda 装很久**。


## 6. 小练习

1. 表里测试 R² 最高的是谁？训练和测试差最大的是谁？
2. 用一句话：为什么看模型不能只看训练分？
3. 若老板只要「好讲的一条直线」，你会推荐谁？若要「稳 + 特征重要性」呢？


In [ ]:
# 在这里写
